# NB2 — Reproducción: Baseline y AAN (BERT-base)

**Plataforma:** Vast.ai (A100) — requiere GPU.

## Propósito
Reproducir los resultados del paper original entrenando dos modelos sobre MR y SemEval:

1. **Baseline** — Fine-tuning completo de BERT-base con una cabeza de clasificación lineal.
   Corresponde a la fila *Baseline* del paper original.

2. **AAN (Auxiliary task with All Negatives)** — BERT-base con la tarea auxiliar de
   supervisión negativa descrita en la Sección 2.2 del paper. Muestreo aleatorio de negativos.
   Corresponde a la fila *AAN* del paper original.

## Detalles del fine-tuning
Se realiza **full fine-tuning**: todos los parámetros del encoder (12 capas transformer +
embeddings) más la cabeza de clasificación se optimizan conjuntamente. No se congela
ninguna capa. Esto es consistente con el enfoque estándar de BERT para clasificación
de texto y con lo descrito en el paper original.

## Configuración (fiel al paper)
- Encoder: `bert-base-uncased`
- Batch size: 16
- n negativos: 4
- Optimizador: Adam con β1=0.999, β2=0.9 (orden no estándar, tal como especifica el paper)
- LR: seleccionado de {1e-5, 3e-5, 5e-5} por validación
- Early stopping: paciencia 10 épocas, máximo 50
- Evaluación: media trimmed de 5 trials (se elimina mejor y peor, se promedia el resto)

## Salidas
- `results/nb2_results.json` — métricas de todos los experimentos
- `checkpoints/bert_{modelo}_{dataset}/best_model.pt` — mejor checkpoint por experimento

**Prerequisito:** NB1 debe haberse ejecutado.

## 1. Rutas e importaciones

In [ ]:
import os
import sys
# Apuntamos al caché local para no re-descargar modelos
os.environ['HF_HOME'] = '/workspace/hf_cache'

from pathlib import Path
BASE_DIR = Path('/workspace/negative_supervision')
DATA_DIR = BASE_DIR / 'data'
CKPT_DIR = BASE_DIR / 'checkpoints'
RES_DIR  = BASE_DIR / 'results'
for d in [CKPT_DIR, RES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / 'label_info.json').exists(), 'Ejecutar NB1 primero.'
print(f'Base: {BASE_DIR}')

In [ ]:
!{sys.executable} -m pip install -q transformers==4.40.0 datasets==2.19.0 scikit-learn==1.4.2 pandas numpy sentencepiece protobuf

In [ ]:
import json, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def set_seed(seed):
    """Fija semilla para reproducibilidad en todas las librerías."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

with open(DATA_DIR / 'label_info.json') as f:
    LABEL_INFO = json.load(f)

print('Importaciones OK.')

## 2. Clases de Dataset

Dos clases según el tipo de tarea:
- `SingleLabelDataset`: etiqueta entera para clasificación con softmax (MR)
- `MultiLabelDataset`: vector multi-hot float para clasificación con sigmoid (SemEval)

In [ ]:
class SingleLabelDataset(Dataset):
    """Dataset para clasificación single-label (MR). Etiqueta como entero."""
    def __init__(self, df, tokenizer, max_len=128):
        self.texts, self.labels = df['text'].tolist(), df['label'].tolist()
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class MultiLabelDataset(Dataset):
    """Dataset para clasificación multi-label (SemEval). Etiqueta como vector float."""
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['text'].tolist()
        self.label_vecs = df['label_vec'].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x).tolist()
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.label_vecs[idx], dtype=torch.float)
        }

## 3. Definición de modelos

### BaselineClassifier
Encoder BERT + cabeza lineal. Full fine-tuning estándar.
Función de pérdida: cross-entropy (single-label) o BCE (multi-label).

### AANClassifier
Extiende el baseline con una tarea auxiliar de supervisión negativa (Sección 2.2 del paper).

**Pérdida auxiliar:**
```
La = (1/n) * Σ_j [1 + cosine_similarity(v_anchor, v_neg_j)]
```
Como cosine_similarity ∈ [-1, 1], el término (1 + cos_sim) ∈ [0, 2].
Minimizar La empuja los negativos hacia cos_sim = -1, es decir,
representaciones maximalmente distintas.

**Pérdida total:** L = Lm + La

Los negativos se muestrean aleatoriamente del mismo batch — ítems cuya
etiqueta difiere del ancla. Para single-label: distinto entero; para
multi-label: distinto vector de etiquetas.

In [ ]:
class BaselineClassifier(nn.Module):
    def __init__(self, encoder_name, num_labels, task_type):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(encoder_name)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.task_type  = task_type

    def encode(self, input_ids, attention_mask):
        """Extrae representación [CLS] del encoder."""
        return self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state[:, 0, :]

    def forward(self, input_ids, attention_mask, labels=None):
        v, logits = self.encode(input_ids, attention_mask), None
        logits = self.classifier(v)
        loss = None
        if labels is not None:
            loss = (F.cross_entropy(logits, labels) if self.task_type == 'single_label'
                    else F.binary_cross_entropy_with_logits(logits, labels))
        return loss, logits


class AANClassifier(nn.Module):
    """
    AAN con muestreo ALEATORIO de negativos — reproducción del paper original.

    Pérdida auxiliar (Sección 2.2):
        La = (1/n) * Σ_j [1 + cosine_similarity(v_main, v_neg_j)]
    Pérdida total:
        L = Lm + La
    """
    def __init__(self, encoder_name, num_labels, task_type, n_negatives=4):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        self.classifier  = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.task_type   = task_type
        self.n_negatives = n_negatives

    def encode(self, input_ids, attention_mask):
        return self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state[:, 0, :]

    def auxiliary_loss(self, v, labels):
        """
        Calcula La sobre el batch actual.
        Para cada ancla i, selecciona aleatoriamente n negativos del batch
        (items con distinta etiqueta) y computa la penalización por similitud.
        """
        batch_size = v.size(0)
        total_loss = torch.tensor(0.0, device=v.device)
        count = 0
        # Detenemos gradientes en el pool de negativos — el gradiente solo
        # fluye a través del ancla, como en el diseño original del paper
        v_pool = v.detach()

        for i in range(batch_size):
            # Identificar ítems con etiqueta diferente al ancla
            if self.task_type == 'single_label':
                diff_mask = (labels != labels[i])
            else:
                diff_mask = ~(labels == labels[i].unsqueeze(0)).all(dim=1)

            diff_indices = diff_mask.nonzero(as_tuple=True)[0]
            if len(diff_indices) == 0:
                continue

            # Muestreo ALEATORIO de hasta n negativos
            n       = min(self.n_negatives, len(diff_indices))
            perm    = torch.randperm(len(diff_indices), device=v.device)[:n]
            neg_idx = diff_indices[perm]

            v_anc      = v[i].unsqueeze(0).expand(n, -1)
            cos_sim    = F.cosine_similarity(v_anc, v_pool[neg_idx], dim=1)
            total_loss = total_loss + (1.0 + cos_sim).mean()
            count     += 1

        if count == 0:
            return torch.tensor(0.0, device=v.device, requires_grad=True)
        return total_loss / count

    def forward(self, input_ids, attention_mask, labels=None):
        v      = self.encode(input_ids, attention_mask)
        logits = self.classifier(v)
        loss   = None
        if labels is not None:
            lm   = (F.cross_entropy(logits, labels) if self.task_type == 'single_label'
                    else F.binary_cross_entropy_with_logits(logits, labels))
            loss = lm + self.auxiliary_loss(v, labels)
        return loss, logits

## 4. Métricas

In [ ]:
def accuracy(logits, labels):
    """Accuracy para clasificación single-label."""
    return (logits.argmax(dim=1) == labels).float().mean().item()

def exact_match(logits, labels, threshold=0.5):
    """
    Exact match para clasificación multi-label (Ecuación 1 del paper).
    Una predicción es correcta solo si TODAS las etiquetas coinciden exactamente.
    """
    preds = (torch.sigmoid(logits) >= threshold).float()
    return (preds == labels).all(dim=1).float().mean().item()

def compute_metric(logits, labels, task_type):
    return accuracy(logits, labels) if task_type == 'single_label' \
           else exact_match(logits, labels)

## 5. Bucles de entrenamiento y evaluación

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    """Una época de entrenamiento con gradient clipping."""
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        loss, _ = model(input_ids, attn_mask, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, task_type):
    """Evaluación sobre un loader completo."""
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        _, logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_logits.append(logits)
        all_labels.append(batch['label'].to(DEVICE))
    return compute_metric(torch.cat(all_logits), torch.cat(all_labels), task_type)


def run_single_trial(model_class, model_kwargs, train_loader, val_loader,
                     test_loader, task_type, lr, seed, patience=10, max_epochs=50):
    """
    Un trial completo: entrenamiento con early stopping + evaluación en test.

    Early stopping: detiene el entrenamiento cuando la métrica de validación
    no mejora por `patience` épocas consecutivas. El modelo se restaura al
    mejor checkpoint antes de evaluar en test.

    Retorna: (test_metric, val_metric, modelo_entrenado)
    """
    set_seed(seed)
    model = model_class(**model_kwargs).to(DEVICE)

    # Adam con β1=0.999, β2=0.9 — orden no estándar especificado en el paper
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.999, 0.9))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * max_epochs * len(train_loader)),
        num_training_steps=max_epochs * len(train_loader)
    )
    best_val, best_state, no_improve = -1.0, None, 0

    for epoch in range(1, max_epochs + 1):
        train_epoch(model, train_loader, optimizer, scheduler)
        val_metric = evaluate(model, val_loader, task_type)
        if val_metric > best_val:
            best_val, best_state, no_improve = val_metric, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f'      Early stop época {epoch} | val={val_metric:.4f}')
            break

    model.load_state_dict(best_state)
    return evaluate(model, test_loader, task_type), best_val, model


def run_experiment(model_class, model_kwargs, train_loader, val_loader,
                   test_loader, task_type, lr, n_trials=5, ckpt_path=None):
    """
    Ejecuta n_trials trials independientes y reporta la media trimmed:
    se elimina el mejor y el peor resultado, se promedia el resto.
    Esta estrategia reduce el efecto de inicializaciones afortunadas o
    desafortunadas, tal como describe el paper original.
    """
    test_scores, best_overall_val, best_state = [], -1.0, None

    for t in range(n_trials):
        print(f'    Trial {t+1}/{n_trials} ...')
        test_m, val_m, model = run_single_trial(
            model_class, model_kwargs, train_loader, val_loader,
            test_loader, task_type, lr, seed=t)
        test_scores.append(test_m)
        print(f'      val={val_m:.4f} | test={test_m:.4f}')
        if val_m > best_overall_val:
            best_overall_val, best_state = val_m, copy.deepcopy(model.state_dict())

    # Media trimmed: eliminar mejor y peor
    trimmed    = sorted(test_scores)[1:-1]
    mean_score = float(np.mean(trimmed))
    std_score  = float(np.std(trimmed))

    if ckpt_path is not None:
        ckpt_path.mkdir(parents=True, exist_ok=True)
        torch.save(best_state, ckpt_path / 'best_model.pt')
        print(f'    Checkpoint guardado → {ckpt_path}')

    print(f'    => Media trimmed: {mean_score:.4f} ± {std_score:.4f}')
    return {'mean': round(mean_score,4), 'std': round(std_score,4),
            'all_test_scores': test_scores, 'trimmed_scores': trimmed}


def load_splits(dataset_name, tokenizer, batch_size=16):
    """Carga los DataLoaders de train/val/test para un dataset."""
    info = LABEL_INFO[dataset_name]
    path = DATA_DIR / dataset_name
    Cls  = SingleLabelDataset if info['task_type'] == 'single_label' else MultiLabelDataset
    return (
        DataLoader(Cls(pd.read_csv(path/'train.csv'), tokenizer),
                   batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True),
        DataLoader(Cls(pd.read_csv(path/'val.csv'),   tokenizer),
                   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
        DataLoader(Cls(pd.read_csv(path/'test.csv'),  tokenizer),
                   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True),
        info['task_type'], info['num_labels']
    )


def select_lr(model_class, model_kwargs, train_loader, val_loader,
              test_loader, task_type, candidates=[1e-5, 3e-5, 5e-5]):
    """Selecciona el mejor LR ejecutando un trial por candidato."""
    best_lr, best_val = None, -1.0
    for lr in candidates:
        print(f'    LR={lr} ...')
        _, val_m, _ = run_single_trial(model_class, model_kwargs,
                                       train_loader, val_loader, test_loader,
                                       task_type, lr, seed=99)
        print(f'      val={val_m:.4f}')
        if val_m > best_val:
            best_val, best_lr = val_m, lr
    print(f'    Mejor LR: {best_lr}')
    return best_lr

## 6. Ejecutar experimentos

2 datasets × 2 modelos × (3 trials de selección de LR + 5 trials completos) = 16 runs.
Los resultados se guardan incrementalmente tras cada experimento.

In [ ]:
ENCODER_NAME = 'bert-base-uncased'
tokenizer    = AutoTokenizer.from_pretrained(ENCODER_NAME)
NB2_RESULTS  = {}

for dataset_name in ['mr', 'semeval']:
    print(f'\n{"="*60}')
    print(f'DATASET: {dataset_name.upper()}')
    print(f'{"="*60}')

    train_loader, val_loader, test_loader, task_type, num_labels = \
        load_splits(dataset_name, tokenizer)
    NB2_RESULTS[dataset_name] = {}

    for model_name, model_class, extra in [
        ('baseline', BaselineClassifier, {}),
        ('aan',      AANClassifier,      {'n_negatives': 4})
    ]:
        print(f'\n  --- {model_name.upper()} ---')
        model_kwargs = {'encoder_name': ENCODER_NAME,
                        'num_labels': num_labels, 'task_type': task_type, **extra}

        print('  Seleccionando LR...')
        best_lr = select_lr(model_class, model_kwargs, train_loader,
                            val_loader, test_loader, task_type)

        print(f'  Ejecutando 5 trials con LR={best_lr}...')
        result = run_experiment(model_class, model_kwargs, train_loader,
                                val_loader, test_loader, task_type, best_lr,
                                n_trials=5,
                                ckpt_path=CKPT_DIR/f'bert_{model_name}_{dataset_name}')
        result['best_lr'] = best_lr
        NB2_RESULTS[dataset_name][model_name] = result

        with open(RES_DIR / 'nb2_results.json', 'w') as f:
            json.dump(NB2_RESULTS, f, indent=2)
        print('  Guardado.')

## 7. Resumen

In [ ]:
with open(RES_DIR / 'nb2_results.json') as f:
    NB2_RESULTS = json.load(f)

print('Resultados NB2 — BERT-base')
print(f'{"Dataset":<12} {"Modelo":<12} {"Métrica":<14} {"Media":>8} {"Std":>8} {"LR":>10}')
print('-' * 62)
for ds in ['mr', 'semeval']:
    metric = LABEL_INFO[ds]['metric']
    for m in ['baseline', 'aan']:
        r = NB2_RESULTS[ds][m]
        print(f'{ds:<12} {m:<12} {metric:<14} {r["mean"]:>8.4f} {r["std"]:>8.4f} {str(r["best_lr"]):>10}')
print()
print('NB2 completo. Continuar con NB3.')